# Lab 4: Fault Locking, Creep, and the Earthquake Cycle

> **Colab note:** This notebook is designed to run on **Google Colab**.

## Introduction

In lecture we followed deformation through the full earthquake cycle — from slow interseismic strain accumulation, through coseismic slip, to postseismic relaxation. In this lab you will work with real geodetic data from two California fault systems to see each phase directly.

In the first half you will fit the Savage–Burford interseismic model to GNSS velocity profiles across the San Andreas, compare a locked segment to a creeping one, and convert the inferred slip deficit into an earthquake moment budget. In the second half you will work with coseismic GNSS offsets from the 2019 Ridgecrest M7.1 earthquake and postseismic time series to identify the mechanisms of postseismic relaxation.

## Learning objectives

- Rotate GNSS velocities into fault-parallel and fault-perpendicular components
- Fit the Savage–Burford arctangent model and interpret slip rate and locking depth
- Distinguish broad elastic strain accumulation from localized fault creep
- Convert a slip deficit into an earthquake moment budget without treating it as a prediction
- Map and model coseismic GNSS offsets and fit the 1D coseismic dislocation model
- Identify afterslip in a postseismic GNSS time series and fit a logarithmic decay model

## Outline

**Part I: Interseismic**
1. [Setup](#1-setup)
2. [Load and standardize the velocity table](#2-load-and-standardize-the-velocity-table)
3. [Define San Andreas coordinates](#3-define-san-andreas-coordinates)
4. [Guided example: the Carrizo segment](#4-guided-example-the-carrizo-segment)
5. [Fit the arctangent locking model](#5-fit-the-arctangent-locking-model)
6. [Parameter tradeoffs](#6-parameter-tradeoffs)
7. [Creep: locked versus creeping behavior](#7-creep-locked-versus-creeping-behavior)
8. [Parkfield: a creeping segment](#8-parkfield-a-creeping-segment)
9. [Slip deficit and moment budget](#9-slip-deficit-and-moment-budget)

**Part II: Coseismic**

10. [Ridgecrest M7.1 coseismic offsets](#10-ridgecrest-m71-coseismic-offsets)
11. [Fit the 1D coseismic dislocation model](#11-fit-the-1d-coseismic-dislocation-model)

**Part III: Postseismic**

12. [Ridgecrest postseismic time series](#12-ridgecrest-postseismic-time-series)
13. [Fit a logarithmic afterslip model](#13-fit-a-logarithmic-afterslip-model)

14. [Synthesis](#14-synthesis)

Archived velocity data: [Kreemer et al. (2022)](https://doi.org/10.7910/DVN/BICMWB)  
Coseismic offsets: [Nevada Geodetic Laboratory, UNR](https://geodesy.unr.edu/news_items/20190707/ci38457511_forweb.txt)  
Postseismic time series: [Nevada Geodetic Laboratory tenv3 archive](http://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/)


## 1. Setup

The notebook first looks for a local course copy of the Kreemer velocity table. If none is present, it attempts to download the file from Harvard Dataverse.


In [ ]:
import io
import re
import zipfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_URL = "https://dataverse.harvard.edu/api/access/datafile/6282416"
LOCAL_CANDIDATES = [
    Path("../data/kreemer_2022_gps_velocities.txt"),
    Path("data/kreemer_2022_gps_velocities.txt"),
    Path("kreemer_2022_gps_velocities.txt"),
]


## 2. Load and standardize the velocity table

The archive may use names such as `Ve`, `Evel`, or `east_velocity` for the same quantity. The next cells parse the table and map its columns onto a common set of names.


In [ ]:
def read_velocity_bytes(raw):
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            members = [n for n in archive.namelist() if not n.endswith("/")]
            preferred = [n for n in members if n.lower().endswith((".txt", ".csv", ".tsv", ".dat"))]
            member = (preferred or members)[0]
            raw = archive.read(member)
            print(f"Reading {member} from downloaded archive")

    for kwargs in [
        dict(sep=None, engine="python", comment="#"),
        dict(sep=r"\s+", engine="python", comment="#"),
    ]:
        try:
            table = pd.read_csv(io.BytesIO(raw), **kwargs)
            if table.shape[1] >= 4:
                return table
        except Exception:
            pass
    raise ValueError("The file could not be parsed as a velocity table.")


local_file = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
if local_file is not None:
    print(f"Loading local data: {local_file}")
    raw = local_file.read_bytes()
else:
    print("No local copy found; downloading from Harvard Dataverse...")
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            raw = response.read()
    except Exception as error:
        raise RuntimeError(
            "Automatic download failed. Download Dataverse file 6282416 in a browser, "
            "save it as data/kreemer_2022_gps_velocities.txt, and rerun this cell."
        ) from error

velocity_raw = read_velocity_bytes(raw)
print(f"Rows: {len(velocity_raw):,}; columns: {velocity_raw.shape[1]}")
display(velocity_raw.head())
print("Column names:", list(velocity_raw.columns))


In [ ]:
COLUMN_MAP = {
    "station": None,
    "lon": None,
    "lat": None,
    "ve": None,
    "vn": None,
    "se": None,
    "sn": None,
}

ALIASES = {
    "station": ["station", "site", "sta", "name", "code", "id"],
    "lon": ["longitude", "lon", "long"],
    "lat": ["latitude", "lat"],
    "ve": ["ve", "v_e", "eastvelocity", "east_vel", "evel", "vel_e", "east"],
    "vn": ["vn", "v_n", "northvelocity", "north_vel", "nvel", "vel_n", "north"],
    "se": ["se", "sve", "sig_e", "sigma_e", "east_sigma", "east_unc", "e_unc"],
    "sn": ["sn", "svn", "sig_n", "sigma_n", "north_sigma", "north_unc", "n_unc"],
}


def canonical(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def infer_column(columns, aliases):
    normalized = {canonical(c): c for c in columns}
    for alias in aliases:
        if canonical(alias) in normalized:
            return normalized[canonical(alias)]
    for alias in aliases:
        matches = [original for key, original in normalized.items()
                   if key.startswith(canonical(alias))]
        if len(matches) == 1:
            return matches[0]
    return None


for key in COLUMN_MAP:
    if COLUMN_MAP[key] is None:
        COLUMN_MAP[key] = infer_column(velocity_raw.columns, ALIASES[key])

print("Detected column mapping:")
for key, value in COLUMN_MAP.items():
    print(f"  {key:>7s} <- {value}")

missing = [key for key, value in COLUMN_MAP.items() if value is None]
if missing:
    raise KeyError(f"Could not identify {missing}. Edit COLUMN_MAP using the printed columns.")

velocity = velocity_raw[[COLUMN_MAP[k] for k in COLUMN_MAP]].copy()
velocity.columns = list(COLUMN_MAP)
for column in ["lon", "lat", "ve", "vn", "se", "sn"]:
    velocity[column] = pd.to_numeric(velocity[column], errors="coerce")
velocity = velocity.dropna(subset=["lon", "lat", "ve", "vn", "se", "sn"]).copy()
velocity["station"] = velocity.station.astype(str)

VELOCITY_SCALE_TO_MM_PER_YR = 1.0  # Change if the archive uses m/yr.
for column in ["ve", "vn", "se", "sn"]:
    velocity[column] *= VELOCITY_SCALE_TO_MM_PER_YR

display(velocity.head())
print(f"Usable velocities: {len(velocity):,}")
print(velocity[["ve", "vn"]].describe())


### Data check

Confirm from the archive metadata that:

1. `ve` and `vn` are the velocities used in the analysis, rather than the postseismic correction terms.
2. The velocities are North America-fixed.
3. The values are in millimeters per year.

Why would fitting uncorrected postseismic velocities bias an interseismic locking model?


## 3. Define San Andreas coordinates

We approximate a short fault segment as a straight line with strike $lpha$, measured clockwise from north. The origin is a point on the fault.

We define positive fault-perpendicular distance toward the **left side** of the fault when looking along strike. For the northwest-striking San Andreas, this is approximately toward the Pacific side.


In [ ]:
EARTH_RADIUS_KM = 6371.0


def geographic_offsets_km(lon, lat, origin_lon, origin_lat):
    """Small-distance east and north offsets from an origin."""
    east = EARTH_RADIUS_KM * np.cos(np.deg2rad(origin_lat)) * np.deg2rad(np.asarray(lon) - origin_lon)
    north = EARTH_RADIUS_KM * np.deg2rad(np.asarray(lat) - origin_lat)
    return east, north


def rotate_to_fault_coordinates(table, origin_lon, origin_lat, strike_deg):
    """Add fault-parallel/perpendicular positions and velocities."""
    result = table.copy()
    east, north = geographic_offsets_km(result.lon, result.lat, origin_lon, origin_lat)
    strike = np.deg2rad(strike_deg)

    # Parallel is positive along strike. Perpendicular is positive to the left.
    result["x_parallel_km"] = east*np.sin(strike) + north*np.cos(strike)
    result["x_perp_km"] = -east*np.cos(strike) + north*np.sin(strike)
    result["v_parallel"] = result.ve*np.sin(strike) + result.vn*np.cos(strike)
    result["v_perp"] = -result.ve*np.cos(strike) + result.vn*np.sin(strike)
    result["s_parallel"] = np.sqrt(
        (result.se*np.sin(strike))**2 + (result.sn*np.cos(strike))**2
    )
    positive_sigma = result.loc[result.s_parallel > 0, "s_parallel"]
    if len(positive_sigma):
        result.loc[result.s_parallel <= 0, "s_parallel"] = positive_sigma.median()
    return result


def add_state_outlines(ax, state_codes=("CA", "NV")):
    try:
        from bokeh.sampledata.us_states import data as states
        for code in state_codes:
            ax.plot(states[code]["lons"], states[code]["lats"], color="0.4", lw=0.8, zorder=0)
    except Exception:
        pass


### Predict the transformation

For a station moving northwest approximately parallel to the San Andreas:

1. Will most of its motion appear in $v_{\parallel}$ or $v_{\perp}$?
2. Does rotating the axes change the station's actual velocity magnitude?
3. Why is a narrow along-strike sampling window important for a one-dimensional profile?


## 4. Guided example: the relatively locked Carrizo segment

The values below approximate a straight section of the San Andreas. The selection includes stations within a prescribed along-strike and cross-fault distance. Adjust the windows if the map reveals poor coverage.


In [ ]:
CARRIZO = {
    "origin_lon": -120.15,
    "origin_lat": 35.25,
    "strike_deg": 320.0,
    "along_halfwidth_km": 120.0,
    "cross_halfwidth_km": 180.0,
}

carrizo_all = rotate_to_fault_coordinates(
    velocity,
    CARRIZO["origin_lon"], CARRIZO["origin_lat"], CARRIZO["strike_deg"]
)
carrizo = carrizo_all[
    carrizo_all.x_parallel_km.abs().le(CARRIZO["along_halfwidth_km"])
    & carrizo_all.x_perp_km.abs().le(CARRIZO["cross_halfwidth_km"])
].copy()

print(f"Selected {len(carrizo)} stations")
display(carrizo[["station", "lon", "lat", "x_perp_km", "x_parallel_km", "v_parallel"]].head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
add_state_outlines(ax)
ax.scatter(carrizo_all.lon, carrizo_all.lat, s=8, color="0.8", label="other stations")
ax.scatter(carrizo.lon, carrizo.lat, s=25, color="#2166ac", label="profile selection")

# Plot the idealized straight fault segment through the origin.
strike = np.deg2rad(CARRIZO["strike_deg"])
lengths = np.array([-150, 150])
fault_e = lengths*np.sin(strike)
fault_n = lengths*np.cos(strike)
fault_lon = CARRIZO["origin_lon"] + np.rad2deg(fault_e/(EARTH_RADIUS_KM*np.cos(np.deg2rad(CARRIZO["origin_lat"]))))
fault_lat = CARRIZO["origin_lat"] + np.rad2deg(fault_n/EARTH_RADIUS_KM)
ax.plot(fault_lon, fault_lat, color="#b2182b", lw=2.5, label="idealized SAF")
ax.set(xlim=(-122.5, -118.0), ylim=(33.5, 37.2), xlabel="Longitude", ylabel="Latitude",
       title="Stations selected for the Carrizo profile")
ax.set_aspect(1/np.cos(np.deg2rad(35)))
ax.legend(frameon=False)

ax = axes[1]
ax.errorbar(carrizo.x_perp_km, carrizo.v_parallel, yerr=carrizo.s_parallel,
            fmt="o", ms=4, color="#2166ac", ecolor="0.65", alpha=0.85)
ax.axvline(0, color="#b2182b", ls="--", lw=1.5, label="idealized fault")
ax.set(xlabel="Fault-perpendicular distance (km; positive toward Pacific side)",
       ylabel="Fault-parallel velocity (mm/yr)", title="Observed fault-parallel velocity profile")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


### Inspect before fitting

1. Does the velocity profile resemble an arctangent?
2. What approximate far-field velocity difference do you see?
3. Are the two sides sampled equally well?
4. Identify any stations that appear inconsistent with the overall trend. Should they automatically be removed?


## 5. Fit the arctangent locking model

We fit

$$
v_{\parallel}(x)=c+\frac{V}{\pi}\tan^{-1}\left(\frac{x}{D}\right).
$$

The model parameters are the relative slip rate $V$, locking depth $D$, and velocity offset $c$.


In [ ]:
def arctan_velocity(x_km, slip_rate_mm_yr, locking_depth_km, offset_mm_yr):
    return offset_mm_yr + slip_rate_mm_yr/np.pi * np.arctan(x_km/locking_depth_km)


initial_guess = [35.0, 15.0, np.median(carrizo.v_parallel)]
bounds = ([0.0, 1.0, -100.0], [100.0, 80.0, 100.0])

parameters, covariance = curve_fit(
    arctan_velocity,
    carrizo.x_perp_km,
    carrizo.v_parallel,
    p0=initial_guess,
    sigma=carrizo.s_parallel,
    absolute_sigma=True,
    bounds=bounds,
)
parameter_sigma = np.sqrt(np.diag(covariance))
V_fit, D_fit, c_fit = parameters

results = pd.DataFrame({
    "estimate": parameters,
    "formal_sigma": parameter_sigma,
    "units": ["mm/yr", "km", "mm/yr"],
}, index=["slip rate V", "locking depth D", "velocity offset c"])
display(results)


In [ ]:
x_model = np.linspace(carrizo.x_perp_km.min(), carrizo.x_perp_km.max(), 600)
v_model = arctan_velocity(x_model, *parameters)
v_predicted = arctan_velocity(carrizo.x_perp_km, *parameters)
residual = carrizo.v_parallel - v_predicted

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
axes[0].errorbar(carrizo.x_perp_km, carrizo.v_parallel, yerr=carrizo.s_parallel,
                 fmt="o", ms=4, color="#2166ac", ecolor="0.7", label="GNSS")
axes[0].plot(x_model, v_model, color="#b2182b", lw=3, label="best-fitting arctangent")
axes[0].axvline(0, color="0.35", ls="--")
axes[0].set(ylabel="Fault-parallel velocity (mm/yr)", title="Carrizo locking model")
axes[0].legend(frameon=False)

axes[1].axhline(0, color="0.35", lw=1)
axes[1].scatter(carrizo.x_perp_km, residual, s=22, color="#2166ac")
axes[1].set(xlabel="Fault-perpendicular distance (km)", ylabel="Residual\n(mm/yr)")
plt.tight_layout()
plt.show()

print(f"RMS residual: {np.sqrt(np.mean(residual**2)):.2f} mm/yr")


### Interpret the model

1. Report the estimated slip rate and locking depth.
2. Which feature of the curve constrains slip rate? Which constrains locking depth?
3. Are the formal parameter uncertainties believable given the residual pattern?
4. Do residuals vary randomly, or are nearby stations systematically above or below the model?
5. Name two physical assumptions of the model that are violated by the real San Andreas system.


## 6. Parameter tradeoffs

The following grid holds the offset at its fitted value and calculates weighted misfit for combinations of slip rate and locking depth. A long, narrow minimum indicates that multiple parameter combinations fit the data nearly equally well — a preview of the non-uniqueness problem we will study in the inversion lecture.


In [ ]:
V_grid = np.linspace(max(5, V_fit-20), V_fit+20, 100)
D_grid = np.linspace(2, min(60, D_fit+35), 100)
misfit = np.empty((len(D_grid), len(V_grid)))

for i, D_test in enumerate(D_grid):
    for j, V_test in enumerate(V_grid):
        prediction = arctan_velocity(carrizo.x_perp_km, V_test, D_test, c_fit)
        misfit[i, j] = np.sum(((carrizo.v_parallel - prediction)/carrizo.s_parallel)**2)

fig, ax = plt.subplots(figsize=(8, 6))
levels = np.nanpercentile(misfit, [1, 2, 5, 10, 20, 40, 70])
contour = ax.contour(V_grid, D_grid, misfit, levels=np.unique(levels), cmap="viridis")
ax.clabel(contour, inline=True, fontsize=8)
ax.plot(V_fit, D_fit, "r*", ms=14, label="best fit")
ax.set(xlabel="Slip rate V (mm/yr)", ylabel="Locking depth D (km)",
       title="Slip rate–locking depth misfit surface")
ax.legend(frameon=False)
plt.show()


### Interpret the tradeoff

1. Does the acceptable region form a compact circle or an elongated valley?
2. If you increase locking depth, how must slip rate change to retain a similar profile?
3. What additional observations could reduce this tradeoff?


## 7. Creep: locked versus creeping behavior

We model partial coupling as the sum of two contributions:

- a **broad locked-fault** term with coupling fraction $C$, producing the arctangent profile
- a **narrow creeping** term with fraction $1-C$, producing a sharp velocity step at the surface

The narrow creeping term uses $\tanh(x / w)$ rather than an ideal step function to avoid a mathematical discontinuity. For a small creep width $w$ (here 1 km), this closely approximates the sudden velocity offset you would see if a fault creeps all the way to the surface.

When $C=1$ the fault is fully locked; when $C=0$ the fault creeps freely at the full plate rate.


In [ ]:
def coupled_velocity(x_km, slip_rate, locking_depth, offset, coupling, creep_width_km=1.0):
    locked = coupling * slip_rate/np.pi * np.arctan(x_km/locking_depth)
    creeping = (1-coupling) * slip_rate/2 * np.tanh(x_km/creep_width_km)
    return offset + locked + creeping


fig, ax = plt.subplots(figsize=(9, 5.5))
for coupling, color in zip([1.0, 0.5, 0.0], ["#2166ac", "#7b3294", "#b2182b"]):
    ax.plot(x_model, coupled_velocity(x_model, V_fit, D_fit, c_fit, coupling),
            lw=3, color=color, label=f"C = {coupling:g}")
ax.set(xlim=(-80, 80), xlabel="Fault-perpendicular distance (km)",
       ylabel="Fault-parallel velocity (mm/yr)",
       title="The same far-field rate can be locked or creeping")
ax.axvline(0, color="0.3", ls="--")
ax.legend(frameon=False)
plt.show()


### Creep interpretation

1. Which curve stores the greatest elastic strain near the fault?
2. Which curve has the largest velocity change immediately at the fault?
3. Do all curves accommodate the same long-term far-field motion?
4. Where would you place GNSS stations to distinguish partial coupling from full locking?
5. Would a station spacing of 50 km resolve a 1-km-wide creeping zone?


## 8. Parkfield: a creeping segment

Parkfield lies near the transition between the creeping central San Andreas and more strongly locked segments to the north and south. In lecture we saw that the Hayward and central SAF show surface creep — here you will see what that looks like in a velocity profile.

Apply the same coordinate rotation and profile extraction to the Parkfield region. Do not assume that a fully locked arctangent model will fit well — think about what surface creep does to the near-fault velocity gradient.


In [ ]:
PARKFIELD = {
    "origin_lon": -120.50,
    "origin_lat": 35.90,
    "strike_deg": 320.0,
    "along_halfwidth_km": 80.0,
    "cross_halfwidth_km": 120.0,
}

parkfield_all = rotate_to_fault_coordinates(
    velocity,
    PARKFIELD["origin_lon"], PARKFIELD["origin_lat"], PARKFIELD["strike_deg"]
)
parkfield = parkfield_all[
    parkfield_all.x_parallel_km.abs().le(PARKFIELD["along_halfwidth_km"])
    & parkfield_all.x_perp_km.abs().le(PARKFIELD["cross_halfwidth_km"])
].copy()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(parkfield.x_perp_km, parkfield.v_parallel, yerr=parkfield.s_parallel,
            fmt="o", ms=4, color="#2166ac", ecolor="0.7")
ax.axvline(0, color="#b2182b", ls="--", label="idealized fault")
ax.set(xlabel="Fault-perpendicular distance (km)", ylabel="Fault-parallel velocity (mm/yr)",
       title="Parkfield-region velocity profile")
ax.legend(frameon=False)
plt.show()


### Compare Carrizo and Parkfield

1. Does Parkfield show a sharper near-fault velocity change?
2. Would changing only locking depth reproduce that shape?
3. What observations would demonstrate surface creep directly?
4. Why is coupling likely to vary along strike rather than having one value for the entire San Andreas?


## 9. From slip deficit to an earthquake moment budget

Assume the fitted Carrizo slip rate is the long-term rate and choose a coupling fraction $C$ and recurrence interval $T$:

$$
S_{deficit}=CVT.
$$

For rupture length $L$, width $W$, and shear modulus $\mu$:

$$
M_0=\mu LWS,
\qquad
M_w=\frac{2}{3}\left(\log_{10}M_0-9.1\right).
$$


In [ ]:
coupling = 1.0
recurrence_years = np.array([50, 100, 150, 250])
rupture_length_km = 300.0
rupture_width_km = 15.0
shear_modulus_pa = 30e9

slip_deficit_m = coupling * (V_fit * 1e-3) * recurrence_years
area_m2 = rupture_length_km*1e3 * rupture_width_km*1e3
moment_nm = shear_modulus_pa * area_m2 * slip_deficit_m
mw = (2/3) * (np.log10(moment_nm) - 9.1)

budget = pd.DataFrame({
    "recurrence_yr": recurrence_years,
    "slip_deficit_m": slip_deficit_m,
    "moment_Nm": moment_nm,
    "Mw_equivalent": mw,
})
display(budget)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(recurrence_years, slip_deficit_m, "o-", color="#2166ac", lw=2.5)
axes[0].set(xlabel="Recurrence interval (yr)", ylabel="Slip deficit (m)",
            title="Accumulated slip deficit")
axes[1].plot(recurrence_years, mw, "o-", color="#b2182b", lw=2.5)
axes[1].set(xlabel="Recurrence interval (yr)", ylabel="$M_w$ equivalent",
            title="Moment-budget equivalent")
plt.tight_layout()
plt.show()


### Interpret the earthquake budget

1. How much slip deficit accumulates over 150 years?
2. Why does doubling recurrence interval not double moment magnitude?
3. Recalculate with $C=0.5$. What changes?
4. List at least three assumptions required to interpret this as one earthquake.
5. Explain why the result is a moment-budget equivalent and not a forecast.


---
# Part II: Coseismic Deformation

## 10. Ridgecrest M7.1 coseismic offsets

The 2019 Ridgecrest M7.1 earthquake produced coseismic GNSS offsets measured at hundreds of stations across southern California. These offsets were determined by the Nevada Geodetic Laboratory from 5-minute ultra-rapid solutions around the time of the earthquake.

In this section you will:
- load and parse the UNR offset file
- map the horizontal displacement vectors
- rotate offsets into fault coordinates
- plot and interpret a fault-perpendicular displacement profile

Archived data: [Nevada Geodetic Laboratory, UNR](https://geodesy.unr.edu/news_items/20190707/ci38457511_forweb.txt)  
Course copy: [datasets/Ridgecrest_coseismic/ci38457511_forweb.txt](https://github.com/amtseismo/EPS166/tree/main/datasets/Ridgecrest_coseismic)

**Event parameters (M7.1, 2019-07-06T03:19:53 UTC):**

| Parameter | Value |
|-----------|-------|
| Epicenter | 35.770°N, 117.599°W |
| Depth | 8 km |
| Strike | ~322° (NW-striking, right-lateral) |
| Mw | 7.1 |


In [ ]:
import io, urllib.request

# Event parameters
EQ_LAT    = 35.770
EQ_LON    = -117.599
EQ_STRIKE = 322.0   # degrees clockwise from north
EQ_DEPTH  = 8.0     # km

# GitHub raw URL for the UNR offset file
COSEISMIC_URL = (
    "https://github.com/amtseismo/EPS166/blob/0ec7623208cddbdb1468d8519030b2125263a67e/datasets/Ridgecrest_GNSS_2019/unr_static_offsets.txt"
)

def load_unr_offsets(url):
    """
    Load UNR coseismic offset file.
    Format: Sta Lon Lat de(m) dn(m) du(m) sde sdn sdu
    Returns a DataFrame with columns in mm.
    """
    with urllib.request.urlopen(url) as r:
        text = r.read().decode()
    lines = [l for l in text.strip().split("\n")
             if l.strip() and not l.startswith("=") and not l.startswith("Sta")]
    records = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 9:
            records.append({
                "station": parts[0],
                "lon":  float(parts[1]),
                "lat":  float(parts[2]),
                "de":   float(parts[3]) * 1000,  # m -> mm
                "dn":   float(parts[4]) * 1000,
                "du":   float(parts[5]) * 1000,
                "sde":  float(parts[6]) * 1000,
                "sdn":  float(parts[7]) * 1000,
                "sdu":  float(parts[8]) * 1000,
            })
    return pd.DataFrame(records)


offsets = load_unr_offsets(COSEISMIC_URL)
print(f"Loaded {len(offsets)} stations")
print(f"Max horizontal offset: {(offsets.de**2 + offsets.dn**2)**0.5 .max():.0f} mm")
display(offsets.head())


In [ ]:
# Filter to stations near the epicenter with significant offsets
horiz = np.sqrt(offsets.de**2 + offsets.dn**2)
near  = offsets[
    (np.abs(offsets.lon - EQ_LON) < 3.0) &
    (np.abs(offsets.lat - EQ_LAT) < 2.5)
].copy()
near["horiz_mm"] = np.sqrt(near.de**2 + near.dn**2)

fig, ax = plt.subplots(figsize=(10, 8))

# Scale factor for quiver arrows
scale = 0.015  # degrees per mm

# All stations as background
ax.scatter(near.lon, near.lat, s=12, color="0.7", zorder=2)

# Displacement vectors — color by horizontal magnitude
sc = ax.quiver(
    near.lon, near.lat,
    near.de * scale, near.dn * scale,
    near.horiz_mm,
    scale=1, scale_units="xy", angles="xy",
    cmap="plasma", clim=(0, near.horiz_mm.quantile(0.98)),
    zorder=3, width=0.003,
)
plt.colorbar(sc, ax=ax, label="Horizontal displacement (mm)")

# Epicenter
ax.plot(EQ_LON, EQ_LAT, "r*", ms=16, zorder=5, label="M7.1 epicenter")

# Approximate fault trace
strike_rad = np.deg2rad(EQ_STRIKE)
lengths = np.array([-0.8, 0.8])
fault_lon = EQ_LON + lengths * np.sin(strike_rad) / np.cos(np.deg2rad(EQ_LAT))
fault_lat = EQ_LAT + lengths * np.cos(strike_rad)
ax.plot(fault_lon, fault_lat, "k-", lw=2, label="Approximate fault trace")

# Scale arrow
ax.quiver(-119.5, 33.8, 100*scale, 0, scale=1, scale_units="xy",
          angles="xy", color="k", width=0.004)
ax.text(-119.5, 33.75, "100 mm", fontsize=9)

ax.set_aspect(1/np.cos(np.deg2rad(EQ_LAT)))
ax.set(xlabel="Longitude", ylabel="Latitude",
       title="Ridgecrest M7.1 coseismic GNSS offsets\n(Nevada Geodetic Laboratory, UNR)")
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()


> **Interpret the offset map:**
> 1. Which station has the largest horizontal offset? Roughly how large is it?
> 2. Describe the overall pattern of displacement vectors — are stations on both sides of the fault moving in the same direction or opposite directions?
> 3. Is the pattern consistent with right-lateral slip on a NW-striking fault? Explain.
> 4. The offsets decay with distance from the fault. Does the decay look more like a sharp step or a broad gradient? How does this compare to the interseismic velocity profile?


## 11. Fit the 1D coseismic dislocation model

In lecture we saw that for a surface-rupturing strike-slip earthquake the fault-parallel displacement follows:

$$
u(x) = \frac{S}{\pi} \arctan\!\left(\frac{D}{x}\right)
$$

where $S$ is the average fault slip and $D$ is the seismogenic depth. Compare this to the interseismic model — the argument of the arctan is flipped: $D/x$ instead of $x/D$.

We rotate the coseismic offsets into fault coordinates and fit this model to the fault-perpendicular displacement profile.


In [ ]:
# Rotate coseismic offsets into fault coordinates
# Reuse the rotate_to_fault_coordinates function from Part I
# Map column names to match: de->ve, dn->vn, sde->se, sdn->sn
offsets_renamed = offsets.rename(columns={"de": "ve", "dn": "vn",
                                           "sde": "se", "sdn": "sn"})
offsets_rot = rotate_to_fault_coordinates(
    offsets_renamed, EQ_LON, EQ_LAT, EQ_STRIKE
)

# Select stations within 200 km along strike and 300 km across
profile = offsets_rot[
    offsets_rot.x_parallel_km.abs().le(200) &
    offsets_rot.x_perp_km.abs().le(300)
].copy()

print(f"Profile stations: {len(profile)}")


def coseismic_arctan(x_km, slip_mm, depth_km):
    """
    1D surface-rupture coseismic displacement model.
    u(x) = (S/pi) * arctan(D/x)
    Note: x=0 gives a discontinuity — we avoid it.
    """
    x = np.where(np.abs(x_km) < 0.1, 0.1 * np.sign(x_km + 1e-9), x_km)
    return (slip_mm / np.pi) * np.arctan(depth_km / x)


# Fit — use fault-parallel displacement (v_parallel)
# Sign convention: positive x = Pacific side (left of fault looking along strike)
from scipy.optimize import curve_fit

p0     = [1000.0, 10.0]          # initial: 1 m slip, 10 km depth
bounds = ([0.0, 1.0], [5000.0, 40.0])

try:
    params, pcov = curve_fit(
        coseismic_arctan,
        profile.x_perp_km,
        profile.v_parallel,
        p0=p0, bounds=bounds,
        sigma=profile.s_parallel,
        absolute_sigma=True,
    )
    perr = np.sqrt(np.diag(pcov))
    S_fit, D_fit_co = params
    print(f"Best-fit slip S  = {S_fit:.0f} ± {perr[0]:.0f} mm")
    print(f"Best-fit depth D = {D_fit_co:.1f} ± {perr[1]:.1f} km")
    print(f"Published slip   ≈ 1500–3000 mm (from InSAR/seismic inversions)")
except Exception as e:
    print(f"Fit failed: {e}")
    S_fit, D_fit_co = 2000.0, 10.0

# Plot
x_co = np.concatenate([np.linspace(-300, -0.5, 500), np.linspace(0.5, 300, 500)])
v_co = coseismic_arctan(x_co, S_fit, D_fit_co)

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(profile.x_perp_km, profile.v_parallel, yerr=profile.s_parallel,
            fmt="o", ms=4, color="#2166ac", ecolor="0.7", label="GNSS offsets", zorder=3)
ax.plot(x_co, v_co, color="#b2182b", lw=2.5,
        label=f"Best fit: S={S_fit:.0f} mm, D={D_fit_co:.1f} km")
ax.axvline(0, color="0.35", ls="--", lw=1.5, label="Fault trace")
ax.axhline(0, color="0.8", lw=0.5)
ax.set(xlabel="Fault-perpendicular distance (km; positive toward Pacific)",
       ylabel="Fault-parallel displacement (mm)",
       title=r"Ridgecrest M7.1 coseismic profile — $u(x) = \frac{S}{\pi}\arctan\!\left(\frac{D}{x}\right)$")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


> **Interpret the coseismic model:**
> 1. How does your best-fit slip $S$ compare to published estimates of ~1.5–3 m average slip from InSAR and seismic inversions? What might explain differences?
> 2. The interseismic model uses $\arctan(x/D)$ and the coseismic model uses $\arctan(D/x)$. Describe in words how the shapes differ and why — which is peaked near the fault and which is broad?
> 3. Look at the residuals. Are there stations that deviate strongly from the 1D model? What physical factors might cause these deviations?
> 4. The 1D model assumes a single uniform slip value. Real fault ruptures have spatially variable slip — how would you expect that to affect the surface displacement pattern compared to what we computed?
> 5. If you integrated the coseismic offsets over the whole rupture area (using the moment budget equation from Part I), what Mw would you recover? Does it agree with the catalog value of 7.1?


---
# Part III: Postseismic Deformation

## 12. Ridgecrest postseismic time series

Following the M7.1 earthquake, GNSS stations near the fault recorded continued motion as the crust responded to the stress change. We use daily position time series from the Nevada Geodetic Laboratory (NGL) tenv3 archive, which provides positions in the IGS14 reference frame.

We load time series for **P595** — the nearest station to the M7.1 epicenter (~13 km), which had the largest coseismic offset (~520 mm east) and should show the clearest postseismic signal.

The tenv3 format provides daily positions. We will:
1. Download and parse the time series
2. Identify the coseismic step and subtract interseismic trend
3. Isolate the postseismic residual
4. Fit a logarithmic afterslip model

Archived data: [NGL tenv3 archive](http://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/)  
Format documentation: [UNR tenv3 format](http://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/0README_tenv3.txt)


In [ ]:
from datetime import datetime, timedelta

# M7.1 origin time
EQ_DATE = datetime(2019, 7, 6)

# Stations to load — P595 is nearest to M7.1, P580 is intermediate distance
POSTSEISMIC_STATIONS = ["P595", "P580"]
NGL_BASE = "http://geodesy.unr.edu/gps_timeseries/tenv3/IGS14/"


def load_ngl_tenv3(station, base_url=NGL_BASE):
    """
    Load a NGL tenv3 daily position time series.

    tenv3 columns (relevant ones):
        col 0: station
        col 2: decimal year
        col 8: east  displacement from reference (m)
        col 10: north displacement from reference (m)
        col 12: up    displacement from reference (m)
        col 14: east  uncertainty (m)
        col 15: north uncertainty (m)
        col 16: up    uncertainty (m)

    Returns DataFrame with columns: year_dec, east_mm, north_mm, up_mm,
    se_mm, sn_mm, su_mm, date.
    """
    url = f"{base_url}{station}.tenv3"
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            text = r.read().decode()
    except Exception as e:
        print(f"  {station}: could not download ({e})")
        return None

    rows = []
    for line in text.strip().split("\n"):
        if line.startswith("*") or len(line) < 10:
            continue
        parts = line.split()
        if len(parts) < 17:
            continue
        try:
            rows.append({
                "year_dec":  float(parts[2]),
                "east_mm":   float(parts[8])  * 1000,
                "north_mm":  float(parts[10]) * 1000,
                "up_mm":     float(parts[12]) * 1000,
                "se_mm":     float(parts[14]) * 1000,
                "sn_mm":     float(parts[15]) * 1000,
                "su_mm":     float(parts[16]) * 1000,
            })
        except (ValueError, IndexError):
            continue

    df = pd.DataFrame(rows)
    # Convert decimal year to datetime
    def decyr_to_date(y):
        yr = int(y)
        frac = y - yr
        return datetime(yr, 1, 1) + timedelta(days=frac * 365.25)
    df["date"] = df.year_dec.apply(decyr_to_date)
    df["days_from_eq"] = (df.date - EQ_DATE).dt.days
    return df


print("Downloading NGL daily time series...")
ts_data = {}
for sta in POSTSEISMIC_STATIONS:
    print(f"  {sta}...", end=" ")
    df = load_ngl_tenv3(sta)
    if df is not None:
        ts_data[sta] = df
        print(f"OK ({len(df)} days, {df.year_dec.min():.1f}–{df.year_dec.max():.1f})")


In [ ]:
def plot_timeseries(df, station, component="east", window_years=(-1, 2)):
    """
    Plot a GNSS position time series centred on the earthquake.

    Parameters
    ----------
    df : pd.DataFrame
        Output from load_ngl_tenv3.
    station : str
        Station name for the title.
    component : str
        'east', 'north', or 'up'.
    window_years : tuple
        Time window relative to earthquake in years.
    """
    col = f"{component}_mm"
    scol = f"s{component[0]}_mm"

    mask = (
        (df.days_from_eq >= window_years[0] * 365) &
        (df.days_from_eq <= window_years[1] * 365)
    )
    sub = df[mask].copy()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.errorbar(sub.days_from_eq, sub[col], yerr=sub[scol],
                fmt="o", ms=3, color="#2166ac", ecolor="0.7", alpha=0.7)
    ax.axvline(0, color="#b2182b", lw=1.5, ls="--", label="M7.1 origin")
    ax.set(xlabel="Days relative to M7.1 (2019-07-06)",
           ylabel=f"{component.capitalize()} position (mm)",
           title=f"{station} — {component.capitalize()} component")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


# Plot east component for P595 — shows coseismic step and postseismic
if "P595" in ts_data:
    plot_timeseries(ts_data["P595"], "P595", component="east", window_years=(-0.5, 1.5))
    plot_timeseries(ts_data["P595"], "P595", component="north", window_years=(-0.5, 1.5))


> **Inspect the time series:**
> 1. Identify the coseismic step in the east component. Roughly how large is it? Does it agree with the offset file from Part II?
> 2. After the coseismic step, does the station continue to move or does it return to its pre-earthquake velocity immediately? Describe what you see in the first few weeks after the earthquake.
> 3. Compare the east and north components. Which shows the larger coseismic offset? Is this consistent with right-lateral slip on a NW-striking fault?
> 4. Approximately how long does the postseismic signal appear to persist before returning to a steady rate?


## 13. Fit a logarithmic afterslip model

In lecture we saw that afterslip — continued aseismic fault slip driven by stress concentrations at the rupture edges — produces a characteristic logarithmic time decay:

$$
u_{\text{post}}(t) = A \log\!\left(1 + \frac{t}{\tau}\right)
$$

where:
- $A$ is the afterslip amplitude (mm)
- $\tau$ is the characteristic decay time (days)
- $t$ is days after the earthquake

To isolate the postseismic signal we need to remove:
1. The coseismic step
2. The interseismic trend (the steady pre-earthquake velocity)

We estimate the interseismic rate from the pre-earthquake portion of the time series and the coseismic offset from the difference between the last pre-earthquake day and the first post-earthquake day.


In [ ]:
def isolate_postseismic(df, component="east",
                         pre_window_days=(-365, -10),
                         post_window_days=(3, 500)):
    """
    Isolate the postseismic displacement by removing the interseismic
    trend and coseismic step.

    Returns
    -------
    t_post : np.array
        Days after earthquake.
    u_post : np.array
        Postseismic displacement in mm.
    pre_rate : float
        Estimated interseismic rate in mm/day.
    co_offset : float
        Estimated coseismic offset in mm.
    """
    col  = f"{component}_mm"
    scol = f"s{component[0]}_mm"

    # Pre-earthquake data
    pre = df[(df.days_from_eq >= pre_window_days[0]) &
             (df.days_from_eq <= pre_window_days[1])].copy()
    # Fit linear trend to pre-earthquake
    pre_fit = np.polyfit(pre.days_from_eq, pre[col], 1)
    pre_rate = pre_fit[0]   # mm/day

    # Post-earthquake data
    post = df[(df.days_from_eq >= post_window_days[0]) &
              (df.days_from_eq <= post_window_days[1])].copy()

    # Predicted position from interseismic trend at t=0
    trend_at_eq = np.polyval(pre_fit, 0)
    # Observed position just after the earthquake (first available day)
    obs_after = post.iloc[0][col]
    co_offset  = obs_after - trend_at_eq

    # Remove trend and coseismic offset from post-seismic data
    t_post = post.days_from_eq.values
    u_post = post[col].values - np.polyval(pre_fit, t_post) - co_offset

    return t_post, u_post, pre_rate, co_offset


def log_afterslip(t, A, tau):
    """Logarithmic afterslip model."""
    return A * np.log1p(t / tau)


if "P595" in ts_data:
    t_post, u_post, pre_rate, co_offset = isolate_postseismic(
        ts_data["P595"], component="east"
    )

    print(f"Estimated interseismic rate: {pre_rate * 365:.1f} mm/yr")
    print(f"Estimated coseismic offset:  {co_offset:.0f} mm")
    print(f"Postseismic window: {t_post[0]:.0f} to {t_post[-1]:.0f} days")

    # Fit logarithmic model
    try:
        popt, pcov = curve_fit(
            log_afterslip, t_post, u_post,
            p0=[50.0, 10.0],
            bounds=([0, 0.1], [500, 500]),
        )
        perr = np.sqrt(np.diag(pcov))
        A_fit, tau_fit = popt
        print(f"\nBest-fit afterslip amplitude A = {A_fit:.1f} ± {perr[0]:.1f} mm")
        print(f"Best-fit decay time tau        = {tau_fit:.1f} ± {perr[1]:.1f} days")
    except Exception as e:
        print(f"Fit failed: {e}")
        A_fit, tau_fit = 50.0, 10.0

    # Plot
    t_model = np.linspace(1, t_post[-1], 500)
    u_model = log_afterslip(t_model, A_fit, tau_fit)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Linear time axis
    axes[0].scatter(t_post, u_post, s=15, color="#2166ac", label="Postseismic residual")
    axes[0].plot(t_model, u_model, color="#b2182b", lw=2,
                 label=f"Log fit: A={A_fit:.0f} mm, τ={tau_fit:.0f} d")
    axes[0].axhline(0, color="0.7", lw=0.5)
    axes[0].set(xlabel="Days after M7.1", ylabel="Postseismic east displacement (mm)",
                title="P595 postseismic residual (linear time)")
    axes[0].legend(frameon=False, fontsize=9)

    # Log time axis
    axes[1].scatter(t_post, u_post, s=15, color="#2166ac")
    axes[1].plot(t_model, u_model, color="#b2182b", lw=2)
    axes[1].set_xscale("log")
    axes[1].set(xlabel="Days after M7.1 (log scale)",
                ylabel="Postseismic east displacement (mm)",
                title="P595 postseismic residual (log time)")

    plt.suptitle(
        r"Afterslip model: $u(t) = A\log\!(1 + t/\tau)$",
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()


> **Interpret the afterslip model:**
> 1. How large is the total afterslip amplitude $A$? How does it compare to the coseismic offset at P595?
> 2. What is the characteristic decay time $\tau$? In physical terms, what does $\tau$ represent?
> 3. On the log time axis, a perfect logarithmic decay would appear as a straight line. Does your residual follow this pattern? Are there deviations — and if so, at what timescales?
> 4. In lecture we discussed three postseismic mechanisms: afterslip, poroelastic rebound, and viscoelastic relaxation. Based on the timescale and amplitude of the signal you fitted, which mechanism does this most likely represent? What observations would you need to distinguish them definitively?
> 5. The postseismic signal at P595 continues for months after the earthquake. If you measured the interseismic velocity of P595 one year after the earthquake, would it equal the true long-term interseismic rate? What bias would this introduce into a locking model?


---
## 14. Synthesis

Write a short paragraph (4–6 sentences) for each question below.

> **1. The complete earthquake cycle at Ridgecrest**  
> You have now seen three phases of the earthquake cycle at a single fault system: interseismic loading on the San Andreas (a proxy for the locked phase), coseismic offsets from the M7.1, and postseismic relaxation at P595. Describe how the deformation pattern — spatial extent, amplitude, and temporal behavior — changes across these three phases. What does each phase tell you that the others cannot?

> **2. The arctan flip**  
> The interseismic velocity profile follows $v(x) = \frac{V_s}{\pi}\arctan(x/D)$ and the coseismic displacement profile follows $u(x) = \frac{S}{\pi}\arctan(D/x)$. Describe in physical terms why the argument of the arctan is flipped between the two cases. What does each parameter ($V_s$ vs $S$, $D$ in each context) represent physically?

> **3. Postseismic contamination of interseismic velocities**  
> Suppose you built a GNSS velocity field for southern California using data collected from 2019–2021 — immediately after the Ridgecrest sequence. How would the postseismic signal affect your estimate of interseismic locking on nearby faults? Would you overestimate or underestimate the locking depth, and why?

### References

- Kreemer, C., Hammond, W. C., and Blewitt, G. (2022). Crustal Strain Rates in the Western United States and Their Relationship with Earthquake Rates. https://doi.org/10.7910/DVN/BICMWB
- Savage, J. C. and Burford, R. O. (1973). Geodetic determination of relative plate motion in central California. *Journal of Geophysical Research*, 78(5), 832–845. https://doi.org/10.1029/JB078i005p00832
- Nevada Geodetic Laboratory, University of Nevada Reno. http://geodesy.unr.edu
- Reid, H. F. (1910). *The Mechanics of the Earthquake*. Carnegie Institution of Washington.
